In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# ============================================================
# WEEK 13 — FUNCTION 4 (RL LENS v8)
#
# What changed vs Week 12 (PCA lens v7):
#  1) Exploration–Exploitation as a Multi-Armed Bandit (MAB):
#     - Treat each candidate-generator "mode" as an arm.
#     - Allocate sampling budget adaptively using UCB + Thompson-style noise.
#     - This makes exploration purposeful (try uncertain arms) and exploitation focused
#       (keep pulling arms that produce high surrogate reward).
#
#  2) Feedback-driven adaptation (Q-value updates):
#     - Within this single run, we do "self-play": we sample candidates, evaluate them
#       with the surrogate (reward proxy), then update each arm's Q estimate.
#     - Q-values are updated with an incremental mean (stable, low-variance).
#
#  3) Convergence speed:
#     - Early rounds: broader exploration across arms (via UCB).
#     - Later rounds: automatically concentrates on the best-performing arm(s).
#
#  4) Still domain-safe: x in [0,1]^4.
#  5) x_next printed to 6 decimals or less only.
# ============================================================

# ----------------------------
# 0) Helpers
# ----------------------------
def clamp01(a):
    return np.minimum(1.0, np.maximum(0.0, a))

def fmt_x6(x):
    return "[" + ", ".join(f"{float(v):.6f}" for v in x) + "]"

def is_duplicate(x, X_existing, tol=1e-6):
    return np.any(np.linalg.norm(X_existing - x, axis=1) < tol)

def norm01(v):
    v = np.asarray(v, dtype=np.float64)
    lo, hi = np.min(v), np.max(v)
    return (v - lo) / (hi - lo + 1e-12)

def nearest_dist(cands, X_existing):
    diff = cands[:, None, :] - X_existing[None, :, :]
    d2 = np.sum(diff * diff, axis=2)
    return np.sqrt(np.min(d2, axis=1) + 1e-12)

def topk_nearest(x, X, y, k=3):
    d = np.linalg.norm(X - x[None, :], axis=1)
    idx = np.argsort(d)[:k]
    return idx, d[idx], y[idx]

# ----------------------------
# 1) Input data (Function 4) — RAW
# ----------------------------
X_train_raw = np.array([
    [0.89698105, 0.72562797, 0.17540431, 0.70169437],
    [0.8893564 , 0.49958786, 0.53926886, 0.50878344],
    [0.25094624, 0.03369313, 0.14538002, 0.49493242],
    [0.34696206, 0.0062504 , 0.76056361, 0.61302356],
    [0.12487118, 0.12977019, 0.38440048, 0.2870761 ],
    [0.80130271, 0.50023109, 0.70664456, 0.19510284],
    [0.24770826, 0.06044543, 0.04218635, 0.44132425],
    [0.74670224, 0.7570915 , 0.36935306, 0.20656628],
    [0.40066503, 0.07257425, 0.88676825, 0.24384229],
    [0.6260706 , 0.58675126, 0.43880578, 0.77885769],
    [0.95713529, 0.59764438, 0.76611385, 0.77620991],
    [0.73281243, 0.14524998, 0.47681272, 0.13336573],
    [0.65511548, 0.07239183, 0.68715175, 0.08151656],
    [0.21973443, 0.83203134, 0.48286416, 0.08256923],
    [0.48859419, 0.2119651 , 0.93917791, 0.37619173],
    [0.16713049, 0.87655456, 0.21723954, 0.95980098],
    [0.21691119, 0.16608583, 0.24137226, 0.77006248],
    [0.38748784, 0.80453226, 0.75179548, 0.72382744],
    [0.98562189, 0.66693268, 0.15678328, 0.8565348 ],
    [0.03782483, 0.66485335, 0.16198218, 0.25392378],
    [0.68348638, 0.9027701 , 0.33541983, 0.99948256],
    [0.17034731, 0.75695908, 0.27652049, 0.5312315 ],
    [0.85965692, 0.91959232, 0.20613873, 0.09779683],
    [0.28213837, 0.50598691, 0.53053084, 0.09630162],
    [0.32607578, 0.4723669 , 0.453192  , 0.10588734],
    [0.94838936, 0.89451301, 0.85163782, 0.55219629],
    [0.66495539, 0.04656628, 0.11677747, 0.79371778],
    [0.57776561, 0.42877174, 0.42582587, 0.24900741],
    [0.73861301, 0.48210263, 0.70936644, 0.50397001],
    [0.8548108 , 0.49396462, 0.73530997, 0.80809201],
    [1.085621  , 1.019592  , 1.039177  , 1.099482  ],   # out-of-bounds in raw
    [1.00000e-06, 1.00000e-06, 1.24558e-01, 1.00000e-06],
    [0.866175  , 0.601115  , 0.708072  , 0.020585  ],
    [0.145904  , 0.536548  , 0.6014    , 0.01905   ],
    [0.356293  , 0.442523  , 0.13052   , 0.242559  ],
    [0.061431  , 0.381247  , 0.983792  , 0.705575  ],
    [0.497045, 0.450388, 0.380113, 0.297612],
    [0.480706, 0.444032, 0.354963, 0.354729],
    [0.503198, 0.435617, 0.371338, 0.408316],
    [0.506271, 0.414408, 0.366165, 0.398853],
    [0.492000, 0.438000, 0.345000, 0.370000],
    [0.528417, 0.450340, 0.361593, 0.369060]
], dtype=float)

y_train = np.array([
    -22.10828779, -14.60139663, -11.69993246, -16.05376511, -10.06963343,
    -15.48708254, -12.68168498, -16.02639977, -17.04923465, -12.74176599,
    -27.31639636, -13.52764887, -16.6791152 , -16.50715856, -17.81799934,
    -26.56182083, -12.75832422, -19.44155762, -28.90327367, -13.70274694,
    -29.4270914 , -11.56574199, -26.85778644,  -7.96677535,  -6.70208925,
    -32.62566022, -19.98949793,  -4.02554228, -13.12278233, -23.1394284 ,
    -67.60493430274798, -22.782193418373407, -22.194212794446454, -13.363105653346768,  -5.926020577803715,
    -23.786955955997737, -2.5615259470796796, -0.81371612670717, -1.1642621740684471, -1.2544758182228928,
    -0.9934706551455466, -1.7634797659752084
], dtype=float)

assert len(X_train_raw) == len(y_train), "X_train and y_train length mismatch"

# ----------------------------
# 2) Enforce domain bounds [0,1]^4
# ----------------------------
X_train = clamp01(X_train_raw)

# ----------------------------
# 3) Current best (maximisation)
# ----------------------------
current_best_idx = int(np.argmax(y_train))
current_best_x = X_train[current_best_idx]
current_best_y = float(y_train[current_best_idx])

print("Current best index:", current_best_idx)
print("Current best X (clamped to [0,1]):", current_best_x)
print("Current best y:", current_best_y)

# ----------------------------
# 4) Fixed scaling for [0,1]^4 (raw == scaled)
# ----------------------------
X_scaled = X_train.copy()
y_mean = y_train.mean()
y_std = y_train.std() + 1e-12
y_scaled = (y_train - y_mean) / y_std

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_tensor_all = torch.tensor(X_scaled, dtype=torch.float32, device=device)
y_tensor_all = torch.tensor(y_scaled, dtype=torch.float32, device=device).unsqueeze(-1)

torch.manual_seed(42)
np.random.seed(42)
rng = np.random.default_rng(42)

# ----------------------------
# 5) Surrogate model (ensemble MLP)
# ----------------------------
class MLP(nn.Module):
    def __init__(self, input_dim=4, hidden=(48, 48), p_dropout=0.03):
        super().__init__()
        h1, h2 = hidden
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.ReLU(),
            nn.Dropout(p_dropout),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Dropout(p_dropout),
            nn.Linear(h2, 1),
        )

    def forward(self, x):
        return self.net(x)

def train_model(model, X, y, max_epochs=650, lr=1.2e-3, weight_decay=8e-6, patience=55, min_delta=1e-4):
    criterion = nn.SmoothL1Loss(beta=1.0)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best = float("inf")
    bad = 0
    model.train()
    for _ in range(max_epochs):
        optimizer.zero_grad(set_to_none=True)
        pred = model(X)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        lv = float(loss.item())
        if lv < best - min_delta:
            best = lv
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break
    return best

def train_ensemble(X_tensor, y_tensor, rr, n_ens=5, hidden=(48, 48), dropout=0.03):
    ensemble = []
    n = len(X_tensor)
    for m in range(n_ens):
        boot_idx = rr.integers(0, n, size=n)
        Xb = X_tensor[boot_idx]
        yb = y_tensor[boot_idx]
        torch.manual_seed(500 + m)
        model = MLP(input_dim=4, hidden=hidden, p_dropout=dropout).to(device)
        train_model(model, Xb, yb)
        ensemble.append(model)
    return ensemble

ensemble = train_ensemble(X_tensor_all, y_tensor_all, rng, n_ens=5)

# ----------------------------
# 6) Prediction utilities (original y units) + EI/PI
# ----------------------------
def ensemble_predict(ensemble, X_scaled_tensor):
    preds = []
    with torch.no_grad():
        for model in ensemble:
            model.eval()
            p_scaled = model(X_scaled_tensor).squeeze(-1)
            p_raw = p_scaled * y_std + y_mean
            preds.append(p_raw)
    preds = torch.stack(preds, dim=0)
    return preds.mean(dim=0), preds.std(dim=0) + 1e-9

normal = torch.distributions.Normal(
    torch.tensor(0.0, device=device),
    torch.tensor(1.0, device=device)
)

def expected_improvement(mean, std, best_y, xi=0.004):
    imp = mean - best_y - xi
    Z = imp / std
    ei = imp * normal.cdf(Z) + std * torch.exp(normal.log_prob(Z))
    return torch.where(std > 0, ei, torch.zeros_like(ei))

def probability_of_improvement(mean, std, best_y, xi=0.0):
    imp = mean - best_y - xi
    Z = imp / std
    return normal.cdf(Z)

# ----------------------------
# 7) Weighted PCA (kept) to provide a compact "state" manifold
# ----------------------------
def weighted_pca(X, y, temp=1.8):
    y = np.asarray(y, dtype=np.float64)
    y_max = float(np.max(y))
    w = np.exp((y - y_max) / float(temp))
    w = w / (np.sum(w) + 1e-12)

    mu = np.sum(X * w[:, None], axis=0)
    Xc = X - mu[None, :]
    cov = (Xc * w[:, None]).T @ Xc
    evals, V = np.linalg.eigh(cov)
    order = np.argsort(evals)[::-1]
    evals = evals[order]
    V = V[:, order]
    expl = evals / (np.sum(evals) + 1e-12)
    return mu, V, evals, expl, w

pca_mu, pca_V, pca_evals, pca_expl, pca_w = weighted_pca(X_train, y_train, temp=1.8)

PCA_D = 2
Vd = pca_V[:, :PCA_D]  # (4, d)

def to_pca(x_raw01):
    xc = x_raw01 - pca_mu
    return xc @ Vd

def from_pca(z):
    x = pca_mu + (z @ Vd.T)
    return clamp01(x)

Z_existing = np.array([to_pca(x) for x in X_train], dtype=np.float64)

print("\n=== PCA SUMMARY (carried into Week 13) ===")
print("Explained variance ratio:", [float(v) for v in pca_expl])
print(f"Using PCA_D = {PCA_D} principal components.")
print("Weighted PCA mean:", fmt_x6(pca_mu))
print("PC1 vector:", fmt_x6(pca_V[:, 0]))
print("PC2 vector:", fmt_x6(pca_V[:, 1]))

# ----------------------------
# 8) Week 13 RL Lens: MAB over candidate-generator "arms"
# ----------------------------
DECODING = {"max_tokens": 7000}

# Arms (generator modes)
# - exploit_best_pca: tight around best in PCA coords (exploit)
# - exploit_top_centroid_pca: around weighted centroid (exploit/explore)
# - thompson_pca: sample around top set with stochasticity (explore where uncertain)
# - global_ucb: global uniform with UCB preference (explore)
# - boundary_jitter: push towards boundaries / corners near top points (escape basins)
ARMS = ["exploit_best_pca", "exploit_top_centroid_pca", "thompson_pca", "global_ucb", "boundary_jitter"]

# Bandit controls (exploration–exploitation)
BANDIT = {
    "rounds": 28,              # internal "self-play" rounds
    "batch_per_round": 260,    # candidates sampled per round (adaptive arm choice)
    "ucb_c": 1.30,             # higher -> more exploration
    "thompson_scale": 0.55,    # noise on Q for Thompson component
    "warm_start_each_arm": 1,  # ensure each arm is tried at least once
}

# Candidate perturbation magnitudes (trust region like)
STEP = {
    "pca_sigma_best": 0.060,
    "pca_sigma_top":  0.070,
    "pca_sigma_th":   0.095,
    "orth_sigma":     0.012,
    "global_jitter":  0.000,   # keep global as pure uniform (stable)
    "boundary_eps":   0.030,   # how hard we "snap" to boundaries
}

# Refinement (same idea, but slightly more exploit now)
REFINE = {
    "topk_seed": 220,
    "per_seed": 16,
    "pca_refine": 0.034,
    "orth_refine": 0.007
}

# Scoring weights (a bit more exploit than Week 12)
SCORE_W = {"w_mean": 0.78, "w_ei": 0.18, "w_pi": 0.04}

# Penalties (keep redundancy control)
PEN = {
    "lambda_raw_near": 0.08,
    "lambda_pca_redund": 0.13,
    "min_pca_sep": 0.010
}

def propose_next_point_week13_rl_mab(
    ensemble, X_existing, Z_existing, best_x, rr,
    xi_base=0.0035,           # slightly more exploit than week 12
    dup_tol=1e-6,
    decoding=DECODING,
    bandit=BANDIT,
    step=STEP,
    refine=REFINE,
    weights=SCORE_W,
    pen=PEN,
    top_report=10
):
    # Anchors in PCA space
    z_best = to_pca(best_x.astype(np.float64))
    z_top_centroid = np.sum(Z_existing * pca_w[:, None], axis=0)

    # Define top-set for Thompson-style arm
    topm = max(6, int(0.18 * len(X_existing)))
    top_idx = np.argsort(y_train)[::-1][:topm]
    X_top = X_existing[top_idx]
    Z_top = Z_existing[top_idx]

    V_orth = pca_V[:, PCA_D:]  # (4, 4-d)

    def add_orth_noise(raw, sigma):
        if V_orth.shape[1] == 0 or sigma <= 0:
            return raw
        n = raw.shape[0]
        noise = rr.normal(0.0, sigma, size=(n, V_orth.shape[1])) @ V_orth.T
        return clamp01(raw + noise.astype(np.float32))

    def gen_candidates(arm, n):
        n = int(n)
        if n <= 0:
            return np.zeros((0, 4), dtype=np.float32)

        if arm == "exploit_best_pca":
            z = z_best + rr.normal(0.0, float(step["pca_sigma_best"]), size=(n, PCA_D))
            raw = np.array([from_pca(zz) for zz in z], dtype=np.float32)
            raw = add_orth_noise(raw, float(step["orth_sigma"]))
            return raw

        if arm == "exploit_top_centroid_pca":
            z = z_top_centroid + rr.normal(0.0, float(step["pca_sigma_top"]), size=(n, PCA_D))
            raw = np.array([from_pca(zz) for zz in z], dtype=np.float32)
            raw = add_orth_noise(raw, float(step["orth_sigma"]))
            return raw

        if arm == "thompson_pca":
            # pick random top seed, sample around it with a bit larger sigma
            pick = rr.integers(0, len(Z_top), size=n)
            z0 = Z_top[pick]
            z = z0 + rr.normal(0.0, float(step["pca_sigma_th"]), size=(n, PCA_D))
            raw = np.array([from_pca(zz) for zz in z], dtype=np.float32)
            raw = add_orth_noise(raw, float(step["orth_sigma"]) * 1.15)
            return raw

        if arm == "global_ucb":
            return rr.random((n, 4), dtype=np.float32)

        if arm == "boundary_jitter":
            # choose a top point then "snap" some dims toward 0/1 to escape local basins
            pick = rr.integers(0, len(X_top), size=n)
            base = X_top[pick].astype(np.float32)
            raw = base.copy()
            # decide which dims to snap
            mask = rr.random((n, 4)) < 0.45
            to_one = rr.random((n, 4)) < 0.50
            eps = float(step["boundary_eps"])
            raw[mask & to_one] = 1.0 - eps * rr.random(np.sum(mask & to_one)).astype(np.float32)
            raw[mask & (~to_one)] = eps * rr.random(np.sum(mask & (~to_one))).astype(np.float32)
            # small jitter elsewhere
            raw = clamp01(raw + rr.normal(0.0, 0.010, size=(n, 4)).astype(np.float32))
            return raw

        # fallback
        return rr.random((n, 4), dtype=np.float32)

    def score_candidates(coarse):
        # surrogate
        X_cand_tensor = torch.tensor(coarse, dtype=torch.float32, device=device)
        mean, std = ensemble_predict(ensemble, X_cand_tensor)

        best_y = float(np.max(y_train))

        # dynamic xi: if model uncertainty high, allow more exploration; else exploit
        # (simple rule: xi scales with mean std across batch)
        std_mean = float(std.mean().detach().cpu().item())
        xi = float(xi_base * (0.85 + 0.55 * (std_mean / (abs(best_y) + 1.0))))

        ei = expected_improvement(mean, std, best_y, xi=xi)
        pi = probability_of_improvement(mean, std, best_y, xi=0.0)

        mean_np = mean.detach().cpu().numpy()
        std_np  = std.detach().cpu().numpy()
        ei_np   = ei.detach().cpu().numpy()
        pi_np   = pi.detach().cpu().numpy()

        mean_n = norm01(mean_np)
        ei_n   = norm01(ei_np)
        pi_n   = norm01(pi_np)

        # penalties
        d_raw = nearest_dist(coarse.astype(np.float64), X_existing.astype(np.float64))
        d_raw_n = norm01(d_raw)

        Z_cand = np.array([to_pca(x.astype(np.float64)) for x in coarse], dtype=np.float64)
        dZ = nearest_dist(Z_cand, Z_existing)
        min_sep = float(pen["min_pca_sep"])
        redund = np.maximum(0.0, (min_sep - dZ))
        redund_n = norm01(redund)

        score = (
            float(weights["w_mean"]) * mean_n +
            float(weights["w_ei"])   * ei_n +
            float(weights["w_pi"])   * pi_n -
            float(pen["lambda_raw_near"]) * d_raw_n -
            float(pen["lambda_pca_redund"]) * redund_n
        )

        # reward proxy for bandit: prefer candidates with both high mean and EI
        # (keeps exploration pressure but converges)
        reward_proxy = (0.70 * mean_n + 0.30 * ei_n)

        return {
            "mean": mean_np, "std": std_np, "ei": ei_np, "pi": pi_np,
            "score": score, "reward_proxy": reward_proxy,
            "d_raw": d_raw, "dZ": dZ, "redund": redund
        }

    # --- Bandit state (Q-learning style update with running mean) ---
    K = len(ARMS)
    Q = np.zeros(K, dtype=np.float64)
    N = np.zeros(K, dtype=np.int64)
    total_pulls = 0

    all_cands = []
    all_meta = []

    # Warm start: try each arm at least once
    warm = int(bandit["warm_start_each_arm"])
    for a_i in range(K):
        for _ in range(warm):
            arm = ARMS[a_i]
            c = gen_candidates(arm, n=int(bandit["batch_per_round"] * 0.40))
            if len(c) == 0:
                continue
            s = score_candidates(c)
            # arm reward = max reward proxy in this pull (like best outcome from that policy)
            r = float(np.max(s["reward_proxy"]))
            N[a_i] += 1
            Q[a_i] += (r - Q[a_i]) / float(N[a_i])
            total_pulls += 1
            all_cands.append(c)
            all_meta.append((arm, s))

    # Main bandit self-play rounds
    rounds = int(bandit["rounds"])
    bsz = int(bandit["batch_per_round"])
    c_ucb = float(bandit["ucb_c"])
    th_scale = float(bandit["thompson_scale"])

    for _ in range(rounds):
        # UCB + Thompson hybrid selection
        ucb = np.zeros(K, dtype=np.float64)
        for i in range(K):
            if N[i] == 0:
                ucb[i] = 1e9
            else:
                ucb[i] = Q[i] + c_ucb * np.sqrt(np.log(total_pulls + 1.0) / float(N[i]))
        th = Q + rr.normal(0.0, th_scale / np.sqrt(np.maximum(1, N)), size=K)
        hybrid = 0.58 * ucb + 0.42 * th

        a_i = int(np.argmax(hybrid))
        arm = ARMS[a_i]

        c = gen_candidates(arm, n=bsz)
        s = score_candidates(c)

        r = float(np.max(s["reward_proxy"]))
        N[a_i] += 1
        Q[a_i] += (r - Q[a_i]) / float(N[a_i])
        total_pulls += 1

        all_cands.append(c)
        all_meta.append((arm, s))

    # Combine all candidates from bandit
    coarse = np.vstack(all_cands).astype(np.float32)

    # Evaluate once more on the combined pool for coherent ranking
    scored = score_candidates(coarse)

    # Top-k seeds for refinement
    topk = int(refine["topk_seed"])
    seed_idx = np.argsort(scored["score"])[::-1][:topk]
    seeds = coarse[seed_idx]

    # refinement mainly in PCA space
    per_seed = int(refine["per_seed"])
    pca_ref = float(refine["pca_refine"])
    orth_ref = float(refine["orth_refine"])

    refine_points = []
    for s0 in seeds:
        zs = to_pca(s0.astype(np.float64))
        zj = zs + rr.normal(0.0, pca_ref, size=(per_seed, PCA_D))
        raw_j = np.array([from_pca(z) for z in zj], dtype=np.float32)
        raw_j = add_orth_noise(raw_j, orth_ref)
        refine_points.append(raw_j)

    refine_all = np.vstack([coarse] + refine_points).astype(np.float32)
    scored2 = score_candidates(refine_all)

    # Filter duplicates and pick best-by-score
    keep = []
    for i in range(len(refine_all)):
        if not is_duplicate(refine_all[i], X_existing, tol=dup_tol):
            keep.append(i)
    if len(keep) == 0:
        chosen_i = int(np.argmax(scored2["score"]))
    else:
        keep = np.array(keep, dtype=int)
        chosen_i = int(keep[np.argmax(scored2["score"][keep])])

    def pack(j):
        x = refine_all[j].astype(np.float64)
        return {
            "idx": int(j),
            "x_raw01": x,
            "mean": float(scored2["mean"][j]),
            "std": float(scored2["std"][j]),
            "ei": float(scored2["ei"][j]),
            "pi": float(scored2["pi"][j]),
            "score": float(scored2["score"][j]),
            "d_raw": float(scored2["d_raw"][j]),
            "dZ": float(scored2["dZ"][j]),
            "redund": float(scored2["redund"][j]),
        }

    chosen = pack(chosen_i)

    pool = np.array(keep if len(keep) > 0 else np.arange(len(refine_all)), dtype=int)
    pool_sorted_score = pool[np.argsort(scored2["score"][pool])[::-1]]
    pool_sorted_ei = pool[np.argsort(scored2["ei"][pool])[::-1]]

    report = {
        "top_by_ei": [pack(j) for j in pool_sorted_ei[:top_report]],
        "top_by_score": [pack(j) for j in pool_sorted_score[:top_report]],
    }

    meta = {
        "best_y": float(np.max(y_train)),
        "pca_d": PCA_D,
        "pca_explained": pca_expl,
        "bandit": bandit,
        "bandit_counts": {ARMS[i]: int(N[i]) for i in range(K)},
        "bandit_Q": {ARMS[i]: float(Q[i]) for i in range(K)},
        "selection": "MAB(UCB+Thompson) -> REFINE(PCA) -> GREEDY_BEST_BY_SCORE",
        "domain_bounds": "[0,1]^4 (enforced)",
        "n_candidates_coarse": int(len(coarse)),
        "n_candidates_total": int(len(refine_all)),
    }
    return chosen, report, meta

chosen, report, meta = propose_next_point_week13_rl_mab(
    ensemble=ensemble,
    X_existing=X_train,
    Z_existing=Z_existing,
    best_x=current_best_x.astype(np.float32),
    rr=rng,
    xi_base=0.0035,
    dup_tol=1e-6,
    top_report=10
)

next_x = clamp01(chosen["x_raw01"])
next_mean = chosen["mean"]
next_std  = chosen["std"]
next_ei   = chosen["ei"]
next_pi   = chosen["pi"]
next_score = chosen["score"]

# ----------------------------
# 9) Interpretability add-ons
# ----------------------------
nn_idx, nn_dist, nn_y = topk_nearest(next_x, X_train, y_train, k=3)

# ----------------------------
# 10) Report
# ----------------------------
print("\n================ WEEK 13 FUNCTION 4 RESULTS (v8 — RL LENS) ================")

print("\nCURRENT BEST OBSERVED")
print("x_best =", fmt_x6(current_best_x), ", y_best =", f"{current_best_y:.6f}")

print("\nWEEK 13 SETTINGS (RL lens)")
print("Domain bounds:", meta["domain_bounds"])
print("PCA_D:", meta["pca_d"])
print("Explained variance:", [float(v) for v in meta["pca_explained"]])
print("Bandit rounds:", meta["bandit"]["rounds"])
print("Batch per round:", meta["bandit"]["batch_per_round"])
print("Bandit counts:", meta["bandit_counts"])
print("Bandit Q-values:", {k: round(v, 6) for k, v in meta["bandit_Q"].items()})
print("Selection:", meta["selection"])
print("Total coarse candidates:", meta["n_candidates_coarse"])
print("Total evaluated after refinement:", meta["n_candidates_total"])

print("\nTOP-10 NON-DUPLICATE CANDIDATES (ranked by SCORE used for selection)")
for i, r in enumerate(report["top_by_score"], 1):
    print(
        f"{i:02d}) x={fmt_x6(r['x_raw01'])} | mean={r['mean']:.6f} std={r['std']:.6f} "
        f"EI={r['ei']:.6f} PI={r['pi']:.6f} SCORE={r['score']:.6f} "
        f"d_raw={r['d_raw']:.6f} dZ={r['dZ']:.6f} redund={r['redund']:.6f}"
    )

print("\nRECOMMENDED NEXT POINT (domain-safe; x_next in 6 decimals)")
print("x_next     =", fmt_x6(next_x))
print("mu(x_next) =", f"{next_mean:.6f}")
print("sigma      =", f"{next_std:.6f}")
print("EI         =", f"{next_ei:.6f}")
print("PI         =", f"{next_pi:.6f}")
print("SCORE      =", f"{next_score:.6f}")

print("\nINTERPRETABILITY CHECKS")
print("Nearest observed points to x_next (context):")
for rank, (ii, dd, yy) in enumerate(zip(nn_idx, nn_dist, nn_y), 1):
    print(f"  {rank}) idx={int(ii)}  x={fmt_x6(X_train[int(ii)])}  y={float(yy):.6f}  dist={float(dd):.6f}")

# ----------------------------
# 11) Week 13 reasoning prompts (explicit, RL lens)
# ----------------------------
print("\nWEEK 13 REASONING (RL Lens)")
print("- Exploration–exploitation: shifted from fixed candidate fractions -> a bandit policy that allocates samples to the best-performing generator arms while still trying uncertain arms via UCB/Thompson.")
print("- Feedback/Q-values: each arm gets a reward proxy from surrogate-evaluated candidates; Q updates (running mean) mirror RL value updates, letting sampling concentrate where reward is consistently higher.")
print("- AlphaGo Zero analogy: internal 'self-play' loop repeatedly proposes, evaluates, and updates the sampling policy without external guidance, resembling model-free trial-and-error (bandit learning) guided by a learned value surrogate.")
print("- Real-world benefit: adaptive budget allocation + trust-region refinement improves efficiency (fewer wasted samples) and speeds convergence while maintaining an escape mechanism (boundary/global arms).")

Current best index: 37
Current best X (clamped to [0,1]): [0.480706 0.444032 0.354963 0.354729]
Current best y: -0.81371612670717

=== PCA SUMMARY (carried into Week 13) ===
Explained variance ratio: [0.49410492219288943, 0.30863602444631977, 0.11737190917620094, 0.07988714401244773]
Using PCA_D = 2 principal components.
Weighted PCA mean: [0.497172, 0.437225, 0.361764, 0.363478]
PC1 vector: [-0.373019, 0.129211, 0.083143, -0.915013]
PC2 vector: [0.683801, 0.098411, 0.694274, -0.201779]

================ WEEK 13 FUNCTION 4 RESULTS (v8 — RL LENS) ================

CURRENT BEST OBSERVED
x_best = [0.480706, 0.444032, 0.354963, 0.354729] , y_best = -0.813716

WEEK 13 SETTINGS (RL lens)
Domain bounds: [0,1]^4 (enforced)
PCA_D: 2
Explained variance: [0.49410492219288943, 0.30863602444631977, 0.11737190917620094, 0.07988714401244773]
Bandit rounds: 28
Batch per round: 260
Bandit counts: {'exploit_best_pca': 6, 'exploit_top_centroid_pca': 7, 'thompson_pca': 8, 'global_ucb': 4, 'boundary_jitter